# 06. Classification Basics

## 📚 Learning Objectives

By completing this notebook, you will:
- Train and evaluate classifiers (e.g. sklearn)
- Interpret confusion matrix and metrics
- Compare models and feature importance

## 🔗 Where this fits

**Builds on:** Course 04 (AIAT 114) — Unit 3 (the full classifier line-up) and Course 05 — Unit 4, lesson 05 — several models, one dataset, one confusion matrix each.

**Used later in:** Course 05 — Unit 4, lesson 07, which replaces accuracy with metrics that survive class imbalance.

---


## The Story

**BEFORE**: You know linear regression for continuous predictions, but don't know how to predict categories/classes.

**AFTER**: You'll master classification algorithms (Logistic Regression, Decision Trees) to predict categories instead of continuous values!

**Why this matters**: Classification is essential for real-world problems like spam detection, medical diagnosis, and image recognition!

---

# Unit 4 - Example 6: Classification Basics

## 🔗 Solving the Problem from Example 4

**Remember the dead end from Example 4?**
- We learned linear regression for predicting continuous values
- But we discovered we need to predict categories/classes, not continuous values
- Linear regression doesn't work well for classification problems

**This notebook solves that problem!**
- We'll learn **classification algorithms** (Logistic Regression, Decision Trees, etc.)
- We'll learn how to **predict categories** instead of continuous values
- We'll learn **classification metrics** (accuracy, precision, recall, F1-score)

**This solves the classification problem from Example 4!**

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Features, labels
- sklearn

**Outputs:** What you'll see when you run the cells

- Classifier
- Metrics
- Confusion matrix
- Plots

---

In [1]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, roc_auc_score)

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("=" * 70)
print("Example 6: Classification Basics")
print("=" * 70)
print("\n📚 Prerequisites: Example 4 completed, linear regression knowledge")
print("🔗 This is Example 06 in Unit 4 - classification algorithms")
print("🎯 Goal: Master classification with logistic regression and decision trees")

Example 6: Classification Basics

📚 Prerequisites: Example 4 completed, linear regression knowledge
🔗 This is Example 06 in Unit 4 - classification algorithms
🎯 Goal: Master classification with logistic regression and decision trees


1. CREATE CLASSIFICATION DATA


In [2]:
# WHAT: Load the real Titanic manifest and build the feature table both classifiers will use.
# WHY: A genuinely hard, genuinely noisy problem is the fair playground for comparing classifiers -
#      survival was NOT a clean function of the six recorded columns, and the scores will show it.

print("\n1. Loading Real Classification Data")
print("-" * 70)

DATA_DIR = '../../../Course 04/datasets/raw/'
titanic = pd.read_csv(DATA_DIR + 'titanic.csv')

df = titanic[['Age', 'Fare', 'SibSp', 'Parch', 'Pclass', 'Sex', 'Survived']].copy()
n_missing_age = df['Age'].isna().sum()
df['Age'] = df['Age'].fillna(df['Age'].median())      # 177 real gaps
df['is_female'] = (df['Sex'] == 'female').astype(int)
df = df.drop(columns=['Sex']).rename(columns={'Survived': 'target'})

FEATURES = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass', 'is_female']

print(f"Data shape: {df.shape}")
print(f"Missing ages filled with the median: {n_missing_age} of {len(df)}")
print(f"Target distribution:\n{df['target'].value_counts().to_string()}")
print(f"Positive class rate: {df['target'].mean():.1%}")
print("\nFirst rows:")
print(df.head().to_string())


1. Loading Real Classification Data
----------------------------------------------------------------------
Data shape: (891, 7)
Missing ages filled with the median: 177 of 891
Target distribution:
target
0    549
1    342
Positive class rate: 38.4%

First rows:
    Age     Fare  SibSp  Parch  Pclass  target  is_female
0  22.0   7.2500      1      0       3       0          0
1  38.0  71.2833      1      0       1       1          1
2  26.0   7.9250      0      0       3       1          1
3  35.0  53.1000      1      0       1       1          1
4  35.0   8.0500      0      0       3       0          0


2. LOGISTIC REGRESSION


In [3]:
# WHAT: Split, scale, and fit logistic regression; store test predictions and probabilities.
# WHY: Scaling is fit on the training set only - fitting it on all data would leak test information into training.

print("\n\n2. Logistic Regression")
print("-" * 70)
X_data = df[FEATURES]
y_data = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
logistic_model = LogisticRegression(random_state=42, max_iter=1000)
logistic_model.fit(X_train_scaled, y_train)
y_test_pred_lr = logistic_model.predict(X_test_scaled)
y_test_proba_lr = logistic_model.predict_proba(X_test_scaled)[:, 1]
accuracy_lr = accuracy_score(y_test, y_test_pred_lr)
baseline = max(y_test.mean(), 1 - y_test.mean())

print(f"\nLogistic Regression Accuracy: {accuracy_lr:.4f}")
print(f"Majority-class baseline:      {baseline:.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred_lr):.4f} | "
      f"Recall: {recall_score(y_test, y_test_pred_lr):.4f} | "
      f"F1: {f1_score(y_test, y_test_pred_lr):.4f}")



2. Logistic Regression
----------------------------------------------------------------------

Logistic Regression Accuracy: 0.8045
Majority-class baseline:      0.6145
Precision: 0.7656 | Recall: 0.7101 | F1: 0.7368


3. DECISION TREE


In [4]:
# WHAT: Fit a depth-5 decision tree and print its feature importances.
# WHY: Trees learn non-linear boundaries and explain themselves - importances say which features drove the splits.

print("\n\n3. Decision Tree")
print("-" * 70)
tree_model = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_model.fit(X_train, y_train)     # trees do not need scaling
y_test_pred_dt = tree_model.predict(X_test)
y_test_proba_dt = tree_model.predict_proba(X_test)[:, 1]
accuracy_dt = accuracy_score(y_test, y_test_pred_dt)
print(f"\nDecision Tree Accuracy: {accuracy_dt:.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred_dt):.4f} | "
      f"Recall: {recall_score(y_test, y_test_pred_dt):.4f} | "
      f"F1: {f1_score(y_test, y_test_pred_dt):.4f}")

# Feature importance - which feature drives the tree's decisions?
print("\nFeature importance (decision tree):")
for name, imp in sorted(zip(FEATURES, tree_model.feature_importances_),
                        key=lambda t: -t[1]):
    print(f"  {name:10s}: {imp:.3f}")
print("(importances sum to 1 - higher = more influence on the splits)")
print("\n💡 The tree puts most of its weight on sex - the same variable the")
print("   logistic model gave the largest coefficient. Two very different")
print("   algorithms agreeing is evidence the pattern is in the data, not the model.")



3. Decision Tree
----------------------------------------------------------------------

Decision Tree Accuracy: 0.7598
Precision: 0.7600 | Recall: 0.5507 | F1: 0.6387

Feature importance (decision tree):
  is_female : 0.534
  Pclass    : 0.182
  Fare      : 0.137
  Age       : 0.127
  SibSp     : 0.011
  Parch     : 0.009
(importances sum to 1 - higher = more influence on the splits)

💡 The tree puts most of its weight on sex - the same variable the
   logistic model gave the largest coefficient. Two very different
   algorithms agreeing is evidence the pattern is in the data, not the model.


4. CONFUSION MATRICES


In [5]:
# WHAT: Draw side-by-side confusion matrices for the two models.
# WHY: Reading raw counts of each error type is the honest way to compare classifiers on the same test set.

print("\n\n4. Confusion Matrices")
print("-" * 70)
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
cm_dt = confusion_matrix(y_test, y_test_pred_dt)
print("Logistic Regression:\n", cm_lr)
print("Decision Tree:\n", cm_dt)
labels = ['Died', 'Survived']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices - Titanic survival (test set)')
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=labels, yticklabels=labels)
axes[0].set_title(f'Logistic Regression (acc {accuracy_lr:.3f})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=labels, yticklabels=labels)
axes[1].set_title(f'Decision Tree (acc {accuracy_dt:.3f})')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.savefig('11_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✓ Confusion matrices saved")
print(f"\nFalse negatives (survivors the model wrote off): "
      f"logistic {cm_lr[1, 0]}, tree {cm_dt[1, 0]}")
print(f"False positives (deaths predicted as survivals): "
      f"logistic {cm_lr[0, 1]}, tree {cm_dt[0, 1]}")
plt.close()



4. Confusion Matrices
----------------------------------------------------------------------
Logistic Regression:
 [[95 15]
 [20 49]]
Decision Tree:
 [[98 12]
 [31 38]]


✓ Confusion matrices saved



False negatives (survivors the model wrote off): logistic 20, tree 31
False positives (deaths predicted as survivals): logistic 15, tree 12


5. ROC CURVES


In [6]:
# WHAT: Plot ROC curves and compute AUC for both models.
# WHY: ROC shows the whole threshold trade-off; AUC condenses it into one comparable number.

print("\n\n5. ROC Curves")
print("-" * 70)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_proba_lr)
auc_lr = roc_auc_score(y_test, y_test_proba_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_test_proba_dt)
auc_dt = roc_auc_score(y_test, y_test_proba_dt)
plt.figure(figsize=(10, 6))
plt.plot(fpr_lr, tpr_lr, linewidth=2, label=f'Logistic Regression (AUC = {auc_lr:.4f})')
plt.plot(fpr_dt, tpr_dt, linewidth=2, label=f'Decision Tree (AUC = {auc_dt:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Titanic survival')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('11_roc_curves.png', dpi=300, bbox_inches='tight')
print(f"✓ ROC curves saved")
print(f"AUC - Logistic Regression: {auc_lr:.4f} | Decision Tree: {auc_dt:.4f}")
print("💡 AUC is threshold-free: it asks 'if I pick one survivor and one victim at")
print("   random, how often does the model score the survivor higher?'")
plt.close()



5. ROC Curves
----------------------------------------------------------------------


✓ ROC curves saved
AUC - Logistic Regression: 0.8520 | Decision Tree: 0.8138
💡 AUC is threshold-free: it asks 'if I pick one survivor and one victim at
   random, how often does the model score the survivor higher?'


6. SUMMARY


In [7]:
# WHAT: Print the summary of the model comparison using the measured numbers.
# WHY: A quick recap of models and metrics before the dead-end demo that follows.

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print(f"""
Measured on the real Titanic manifest ({len(df)} passengers, {len(X_test)} held out):

  Majority baseline    accuracy {baseline:.3f}
  Logistic Regression  accuracy {accuracy_lr:.3f}   AUC {auc_lr:.3f}
  Decision Tree (d=5)  accuracy {accuracy_dt:.3f}   AUC {auc_dt:.3f}

Key Concepts Covered:
1. Logistic Regression for classification
2. Decision Tree classifier
3. Confusion matrix analysis (who gets misclassified, and which way)
4. ROC curves and AUC
5. Feature importance from the decision tree

Next Steps: Continue to Example 7 for Model Evaluation
""")


Summary

Measured on the real Titanic manifest (891 passengers, 179 held out):

  Majority baseline    accuracy 0.615
  Logistic Regression  accuracy 0.804   AUC 0.852
  Decision Tree (d=5)  accuracy 0.760   AUC 0.814

Key Concepts Covered:
1. Logistic Regression for classification
2. Decision Tree classifier
3. Confusion matrix analysis (who gets misclassified, and which way)
4. ROC curves and AUC
5. Feature importance from the decision tree

Next Steps: Continue to Example 7 for Model Evaluation



## 🚫 When Classification Hits a Dead End

**BEFORE**: We've learned to build classification models.

**AFTER**: We discover we need proper evaluation beyond just accuracy!

**Why this matters**: Accuracy alone can be misleading - we need comprehensive evaluation metrics!

---

### The Problem We've Discovered

We've learned:
- ✅ How to build classification models (Logistic Regression, Decision Trees)
- ✅ How to calculate accuracy
- ✅ How to create confusion matrices and ROC curves

**But we have a problem:**
- ❓ **What if accuracy is misleading (imbalanced classes)?**
- ❓ **What if we need to understand model performance in detail?**
- ❓ **What if we need to compare multiple models properly?**

**The Dead End:**
- We can build models and calculate accuracy
- But accuracy alone doesn't tell the full story
- We need comprehensive evaluation metrics and techniques

---

### Demonstrating the Problem

Let's see why accuracy alone can be misleading:

In [8]:
# WHAT: Show why accuracy misleads, using a REAL extremely imbalanced dataset: credit-card fraud.
# WHY: 0.17% of these transactions are fraud. A model that never predicts fraud scores 99.8%
#      accuracy and catches nothing - the imbalance is real, not a number we chose.

print("\n" + "=" * 70)
print("🚫 DEMONSTRATING THE DEAD END: Accuracy Can Be Misleading")
print("=" * 70)

# Real data: 284,807 European card transactions from September 2013, with the
# 492 confirmed frauds labelled. (Dal Pozzolo et al., ULB / Worldline.)
fraud = pd.read_csv(DATA_DIR + 'creditcard_fraud.csv',
                    usecols=['V1', 'V2', 'V3', 'V4', 'V10', 'V14', 'Amount', 'Class'])
y_imbalanced = fraud['Class'].values
n_samples = len(fraud)
n_fraud = int(y_imbalanced.sum())

# Dummy classifier that always predicts class 0 (majority class = legitimate)
y_pred_dummy = np.zeros(n_samples)

accuracy_dummy = accuracy_score(y_imbalanced, y_pred_dummy)
print(f"\n📊 Real Imbalanced Dataset - credit-card fraud:")
print(f"   - Total transactions: {n_samples:,}")
print(f"   - Class 0 (legitimate): {n_samples - n_fraud:,} "
      f"({(n_samples - n_fraud)/n_samples*100:.3f}%)")
print(f"   - Class 1 (fraud):      {n_fraud:,} ({n_fraud/n_samples*100:.3f}%)")

print(f"\n⚠️  Dummy Classifier (Always Predicts 'Legitimate'):")
print(f"   - Accuracy: {accuracy_dummy:.4%}")
print(f"   - This looks superb! But the model is useless!")
print(f"   - Recall on fraud: {recall_score(y_imbalanced, y_pred_dummy, zero_division=0):.4f}")
print(f"   - It misses all {n_fraud} frauds - every single one.")

# A real model on the same data, to show the metrics that DO reveal the difference
Xf = fraud.drop(columns=['Class'])
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    Xf, y_imbalanced, test_size=0.3, random_state=42, stratify=y_imbalanced)
fraud_scaler = StandardScaler().fit(Xf_train)
fraud_model = LogisticRegression(max_iter=1000, class_weight='balanced')
fraud_model.fit(fraud_scaler.transform(Xf_train), yf_train)
yf_pred = fraud_model.predict(fraud_scaler.transform(Xf_test))
yf_proba = fraud_model.predict_proba(fraud_scaler.transform(Xf_test))[:, 1]

print(f"\n✅ A real classifier on the same test split "
      f"({len(yf_test):,} transactions, {int(yf_test.sum())} frauds):")
print(f"   - Accuracy:  {accuracy_score(yf_test, yf_pred):.4%}  "
      f"(LOWER than the useless dummy!)")
print(f"   - Precision: {precision_score(yf_test, yf_pred, zero_division=0):.4f}")
print(f"   - Recall:    {recall_score(yf_test, yf_pred, zero_division=0):.4f}  "
      f"<- it actually catches frauds")
print(f"   - F1:        {f1_score(yf_test, yf_pred, zero_division=0):.4f}")
print(f"   - ROC AUC:   {roc_auc_score(yf_test, yf_proba):.4f}")

print(f"\n💡 The Problem:")
print(f"   - The dummy beats the real model on ACCURACY and is worthless")
print(f"   - Only recall, precision, F1 and AUC expose the difference")
print(f"   - Which metric you optimise IS a business decision: a missed fraud")
print(f"     and a blocked legitimate card do not cost the same")

print(f"\n📋 What We Need for Proper Evaluation:")
print(f"   1. Multiple metrics (precision, recall, F1, AUC)")
print(f"   2. Cross-validation (robust performance estimation)")
print(f"   3. Learning curves (understand model behavior)")
print(f"   4. Model comparison (which model is actually better?)")

print(f"\n➡️  Solution Needed:")
print(f"   - We need comprehensive model evaluation techniques")
print(f"   - We need to understand metrics beyond accuracy")
print(f"   - This leads us to Example 7: Model Evaluation")

print("\n" + "=" * 70)


🚫 DEMONSTRATING THE DEAD END: Accuracy Can Be Misleading



📊 Real Imbalanced Dataset - credit-card fraud:
   - Total transactions: 284,807
   - Class 0 (legitimate): 284,315 (99.827%)
   - Class 1 (fraud):      492 (0.173%)

⚠️  Dummy Classifier (Always Predicts 'Legitimate'):
   - Accuracy: 99.8273%
   - This looks superb! But the model is useless!
   - Recall on fraud: 0.0000
   - It misses all 492 frauds - every single one.

✅ A real classifier on the same test split (85,443 transactions, 148 frauds):
   - Accuracy:  97.5586%  (LOWER than the useless dummy!)
   - Precision: 0.0587
   - Recall:    0.8716  <- it actually catches frauds
   - F1:        0.1101
   - ROC AUC:   0.9605

💡 The Problem:
   - The dummy beats the real model on ACCURACY and is worthless
   - Only recall, precision, F1 and AUC expose the difference
   - Which metric you optimise IS a business decision: a missed fraud
     and a blocked legitimate card do not cost the same

📋 What We Need for Proper Evaluation:
   1. Multiple metrics (precision, recall, F1, AUC)
   2. C

### What We Need Next

**The Solution**: We need comprehensive model evaluation:
- **Multiple metrics**: Precision, recall, F1-score, AUC (not just accuracy)
- **Cross-validation**: Robust performance estimation
- **Learning curves**: Understand model behavior and overfitting
- **Model comparison**: Proper techniques to compare models

**This dead end leads us to Example 7: Model Evaluation**
- Example 7 will teach us comprehensive evaluation techniques
- We'll learn metrics beyond accuracy
- We'll learn validation methods to properly assess models!


## 📚 References

1. Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
2. Fawcett, T. (2006). *An Introduction to ROC Analysis*. Pattern Recognition Letters, 27(8), 861-874. <https://doi.org/10.1016/j.patrec.2005.10.010>
3. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*, 2nd ed., Ch. 4 (Classification). Springer. <https://www.statlearning.com>